# 08 — Training Dynamics and Failure Modes

A network can be mathematically correct and still train badly. This notebook visualizes saturation, vanishing/exploding gradients, initialization sensitivity, and overfitting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

## 1. Sigmoid saturation

In [ ]:
z=np.linspace(-10,10,500); s=1/(1+np.exp(-z)); ds=s*(1-s)
plt.figure(figsize=(9,5)); plt.plot(z,s,label='sigmoid'); plt.plot(z,ds,label='derivative'); plt.legend(); plt.grid(alpha=.25); plt.title('Sigmoid saturation shrinks gradients'); plt.show()

When activations saturate, local derivatives approach zero. Repeated multiplication through many layers can then make early-layer gradients extremely small.

## 2. Gradient propagation through depth

In [ ]:
def propagate(depth,weight_scale,activation='tanh'):
    rng=np.random.default_rng(0); grad=np.ones(1000); magnitudes=[]
    for _ in range(depth):
        z=rng.normal(size=1000); w=rng.normal(0,weight_scale,size=1000)
        local=(1-np.tanh(z)**2) if activation=='tanh' else (z>0).astype(float)
        grad=grad*w*local; magnitudes.append(np.mean(np.abs(grad)))
    plt.figure(figsize=(9,4)); plt.plot(range(1,depth+1),magnitudes,marker='o'); plt.yscale('log'); plt.xlabel('layer'); plt.ylabel('mean |gradient|'); plt.title(f'{activation}, weight scale={weight_scale}'); plt.grid(alpha=.25); plt.show()
widgets.interact(propagate,depth=widgets.IntSlider(value=20,min=2,max=60),weight_scale=widgets.FloatSlider(value=1,min=.1,max=2.5,step=.1),activation=widgets.Dropdown(options=['tanh','relu']));

Initialization scale directly affects signal and gradient propagation. This is why Xavier/Glorot and He initializations are principled choices rather than arbitrary defaults.

## 3. Overfitting visualized with a capacity proxy

In [ ]:
rng=np.random.default_rng(4); x=np.linspace(-3,3,25); y=np.sin(x)+rng.normal(0,.18,len(x))
plt.figure(figsize=(9,5)); plt.scatter(x,y,label='training data')
for deg in [1,5,20]:
    coef=np.polyfit(x,y,deg); xx=np.linspace(-3,3,400); yy=np.polyval(coef,xx); plt.plot(xx,yy,label=f'capacity proxy: degree {deg}')
plt.ylim(-2,2); plt.legend(); plt.grid(alpha=.25); plt.title('Too little capacity → underfit; too much → overfit'); plt.show()

Neural-network training balances optimization, architecture, capacity, generalization, data quality, regularization, and initialization. Understanding these dynamics is essential before moving to deeper architectures.